In [1]:
pip install dateparser

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.7/318.7 kB 7.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
from datasets import load_dataset
import re
import dateparser
from datetime import datetime, timedelta
import json
import os

In [3]:
raw_path="/kaggle/input/datasets/chorfidhiaeddine/hotel-conversations-dataset"
conv_datasets = load_dataset(
    "json",
    data_files={
        "valid": f"{raw_path}/hotel_validation_2000.jsonl",
        "train": f"{raw_path}/hotel_conversations_dataset.jsonl",
        "test": f"{raw_path}/hotel_test_2000.jsonl",
    },
)
def resolve_next_this_day(match_text: str):
    text = match_text.lower()
    today = datetime.today()
    days_map = {
        "monday": 0, "tuesday": 1, "wednesday": 2, "thursday": 3,
        "friday": 4, "saturday": 5, "sunday": 6
    }
    for day_name, day_num in days_map.items():
        if day_name in text:
            days_ahead = day_num - today.weekday()
            if days_ahead <= 0:
                days_ahead += 7
            if "next" in text:
                days_ahead += 7
            return today + timedelta(days=days_ahead)
    return None


def normalize_data(text: str):

    date_patterns = [
        r"\b\d{4}[/-]\d{1,2}[/-]\d{1,2}\b",
        r"\b\d{1,2}[/-]\d{1,2}(?:[/-]\d{2,4})?\b",
        r"\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\s\d{1,2}(?:,\s\d{4})?\b",
        r"\b\d{1,2}\s(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\s?\d{0,4}\b",
        r"(?<!next )(?<!this )(?:Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday)(?:\s\d{1,2})?\b",
        r"\b(?:today|tomorrow|day after tomorrow|in \d+ days?|in \d+ weeks?)\b",
        r"\b(?:next|this)\s(?:week|Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday|weekend?)\b",
        r"\b(?:the)\s\d{1,2}\s(?:of)\s(?:January|February|March|April|May|June|July|August|September|October|November|December|Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\b"
    ]
    all_date_matches = []

    day_names = ["monday","tuesday","wednesday","thursday","friday","saturday","sunday"]

    for pattern in date_patterns:
        for m in re.finditer(pattern, text, flags=re.IGNORECASE):
            match = m.group()
            start, end = m.start(), m.end()
            if any(not (end <= s or start >= e) for s, e, _ in all_date_matches):
                continue

            if re.match(r"^\d{4}[/-]", match):
                settings = {"DATE_ORDER": "YMD", "PREFER_DATES_FROM": "future"}
            else:
                settings = {"DATE_ORDER": "DMY", "PREFER_DATES_FROM": "future"}
            if re.match(r"(?i)\b(next|this)\b", match) and \
               any(day in match.lower() for day in day_names):
                dt = resolve_next_this_day(match)
            else:
                dt = dateparser.parse(match, settings=settings)

            if dt:
                normalized_date = dt.strftime("%Y-%m-%d")
                all_date_matches.append((start, end, f" {normalized_date} "))

    for start, end, replacement in sorted(
        all_date_matches, key=lambda x: x[0], reverse=True
    ):
        text = text[:start] + replacement + text[end:]

    price_patterns = [
        r"\b\d+(?:\.\d+)?k\b",
        r"(?<![-/])\b\d+(?:,\d{3})*(?:\.\d+)?\s*\$?\b(?![-/])",
        r"\b\d+(?:\.\d+)?\s*(?:dollars?|usd)\b",
        r"\b\d+(?:\.\d+)?\.00\b",
    ]

    for pattern in price_patterns:
        matches = re.findall(pattern, text, flags=re.IGNORECASE)
        for match in matches:
            num_text = match.lower().strip()
            if "k" in num_text:
                num_text = re.sub(r"k", "", num_text)
                num_value = float(num_text) * 1000
            elif ".00" in num_text:
                num_text = re.sub(r"\.00", "", num_text)
                num_value = float(num_text)
            else:
                num_text = re.sub(r"[^\d\.]", "", num_text)
                num_value = float(num_text)

            normalized_price = str(int(num_value))
            text = re.sub(
                re.escape(match),
                f" {normalized_price} ",
                text,
                flags=re.IGNORECASE,
            )

    text = re.sub(r"\s+", " ", text)
    return text


def process_data(text: str):
    text = text.lower().strip()
    text = re.sub(r"[^a-zA-Z0-9\s\?\!\.\,\:\;\'\"\(\)\-\/\%\$\€]", "", text)
    text = re.sub(r"([^\d+])\1{2,}", r"\1\1", text)
    text = re.sub(r"([!?$.%€])", r" \1 ", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s*[-]\s*", "-", text)
    text = re.sub(
        r"\b(\d{1,4})(?:\s*\.\s*|\s+)(\d{1,2})(?:\s*\.\s*|\s+)(\d{1,4})\b",
        r"\1-\2-\3",
        text,
    )
    text = normalize_data(text)
    return text

def convert_entities_to_bio(text: str, entities: list):
    tokens = text.split()
    labels = ["O"] * len(tokens)
    if not entities:
        return labels
    for entity in entities:
        entity_tokens = process_data(str(entity["value"])).split()
        for i in range(len(tokens)):
            if tokens[i : i + len(entity_tokens)] == entity_tokens:
                labels[i] = f"B-{entity["entity"]}"
                for j in range(1, len(entity_tokens)):
                    labels[i + j] = f"I-{entity["entity"]}"

    return labels
def generate_bert_dataset(dataset, out_path: str):
    with open(out_path, "w", encoding="utf-8") as f:
        for conv in dataset:
            for turn in conv["turns"]:
                if turn["speaker"] == "Guest":
                    procced_utterance = process_data(turn["utterance"])
                    process_turn = {
                        "turn_id": turn["turn_id"],
                        "speaker": turn["speaker"],
                        "utterance": procced_utterance,
                        "intent": turn["intent"],
                        "ner_tags": convert_entities_to_bio(
                            procced_utterance, turn["entities"]
                        ),
                    }
                    f.write(json.dumps(process_turn, ensure_ascii=False) + "\n")


def generate_gpt_dataset(dataset, out_path: str):
    system_content = """
You are a helpful hotel information and reservation assistant.

Your role is to help guests with their questions and requests by using information retrieved from a hotel database.
You provide accurate and concise answers based on the available data.

The guest may ask about hotel availability, prices, facilities, locations, ratings, or general hotel information.
You may also suggest hotels that match the guest's preferences.

You DO NOT perform application operations such as booking rooms, cancelling reservations, or processing payments.
Your role is only to provide information and assistance.

Possible guest intents include:
greet, check_availability, inquire_current_price, inquire_facilities, search_hotel, ask_price_prediction, ask_room_type, ask_breakfast, ask_location, ask_rating, ask_parking, thank_you, goodbye, suggest_hotels, out_of_scope, unintelligible, ask_rooms.

When responding:

- Use the current dialogue state and the guest's request to provide relevant information.
- If required information is missing, politely ask the guest for the necessary details.
- Keep responses natural, clear, and helpful.

Important information that may appear in the dialogue state includes:

- Dates: check_in_date, check_out_date
- Stay details: num_nights, num_guests, num_rooms
- Room information: room_type (single, double, twin, suite, deluxe)
- Hotel attributes: star_rating (3, 4, 5)
- Location: country, city, location (downtown, airport, beach, etc.)
- Hotel identification: hotel_name
- Price information: price_value, price_range (cheap, moderate, expensive), max_price (budget)
- Facilities: wifi, parking, pool, breakfast, gym, etc.

Always use the available information to give the most relevant answer, and ask clarifying questions when needed.
        """
    with open(out_path, "w", encoding="utf-8") as f:

        for conv in dataset:
            process_turn = {"message": []}
            process_turn["message"].append(
                {
                    "role": "system",
                    "content": system_content,
                }
            )
            for i, turn in enumerate(conv["turns"]):
                next_state = (
                    conv["turns"][i + 1]["dialogue_state"]
                    if (i + 1) < len(conv["turns"])
                    else {}
                )

                if turn["speaker"] == "Guest":
                    process_turn["message"].append(
                        {
                            "role": "user",
                            "content": f"Current State: {next_state}\n{turn['utterance']}",
                        }
                    )
                else:
                    process_turn["message"].append(
                        {"role": "assistant", "content": turn["utterance"]}
                    )
            f.write(json.dumps(process_turn, ensure_ascii=False) + "\n")

Generating valid split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [4]:
processed_path= "/kaggle/working/data/processed"
os.makedirs(processed_path,exist_ok=True)
for name, dataset in conv_datasets.items():
    print(f"Start Generate Intent+NER_{name}_dataset======>")
    generate_bert_dataset(
        dataset,
        out_path=f"{processed_path}/convs_intents+NER_{name}.jsonl",
    )
    print("=====>END")
    print(f"Start Generate Dialogue_Mangement_{name}_dataset======>")
    generate_gpt_dataset(
        dataset,
        out_path=f"{processed_path}/convs_dialogues_{name}.jsonl",
    )
    print(f"=====>END")

Start Generate Intent+NER_valid_dataset======>
=====>END
Start Generate Dialogue_Mangement_valid_dataset======>
=====>END
Start Generate Intent+NER_train_dataset======>
=====>END
Start Generate Dialogue_Mangement_train_dataset======>
=====>END
Start Generate Intent+NER_test_dataset======>
=====>END
Start Generate Dialogue_Mangement_test_dataset======>
=====>END
